Install the branch

In [ ]:
!pip install git+https://github.com/phoeenniixx/pytorch-forecasting.git@ptf-new-design

In [1]:
from pytorch_forecasting.data.examples import load_toydata

In [2]:
num_series = 100  # Number of individual time series to generate
seq_length = 50  # Length of each time series
data_df = load_toydata(num_series, seq_length)
data_df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,0.176004,0.286363,0,1.000000,0.000406,0
1,0,1,0.286363,0.465868,0,0.995004,0.000406,0
2,0,2,0.465868,0.571678,0,0.980067,0.000406,0
3,0,3,0.571678,0.586340,0,0.955336,0.000406,0
4,0,4,0.586340,0.856573,0,0.921061,0.000406,0


## Imports

In [3]:
from lightning.pytorch import Trainer
from sklearn.preprocessing import StandardScaler

from pytorch_forecasting.data.encoders import NaNLabelEncoder, TorchNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE
from pytorch_forecasting.models.temporal_fusion_transformer._tft_v2 import TFT
from pytorch_forecasting._proto._tft_forecaster import TFTForecaster
from pytorch_forecasting._proto._timeseries_datatype import TimeSeries_datatype
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule

## 1. Minimal - with `Forecaster`

- `TimeSeries_datatype` is a plain container for the data and its schema, not a `Dataset`.
   - `TimeSeries` will be the actual name, it is taken by D1 layer that is why this name
- Everything not given is inferred: `num`/`cat` from the dtypes, `known`/`static` from the defaults.
- `TFTForecaster()` takes no `metadata`, no datamodule and no trainer.
- `fit` builds the default datamodule, reads its `metadata`, and only then constructs the model.
- `fit` takes a `Trainer`, not trainer keywords. `forecaster.fit(data)` works and uses a default `Trainer`; we pass one here only because that default runs for many epochs (1000 epochs!).

In [4]:
data = TimeSeries_datatype(
    data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
)

forecaster = TFTForecaster()

# `forecaster.fit(data)` is enough - it uses a default `Trainer`. That default
# runs for 1000 epochs, so to keep the notebook quick we pass a small one.
forecaster.fit(data, trainer=Trainer(max_epochs=1))

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable a

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The numbe

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


TFTForecaster()

- `predict` returns a `TimeSeries_datatype`.
- `_series` is the series the forecast was made for, `_time_idx` the time index it falls on, so the result can be joined back to the input. Overlapping windows forecast the same time index more than once.
- The values are in the target normalizer's space, not the original one - the v2 predict path applies no inverse transform. This is something that we still need to add to v2!

In [5]:
preds = forecaster.predict(data)
preds.to_pandas().head()

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecate

Predicting: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


,_series,_time_idx,y
0,0,30,0.565710
1,0,31,0.566749
2,0,32,0.567793
3,0,33,0.568885
4,0,34,0.569958


## 2. Maximal - with `Forecaster`

- The schema is spelled out on the datatype: `num`/`cat`, `known`/`unknown`, `static`.
- The datamodule is passed to `__init__` **configured but without data**; `fit` attaches the data.
- The trainer may be given to `__init__` or to `fit`; the one in `fit` wins.
- `predict(new_data)` reuses the transforms fitted in `fit`, it does not refit them.

In [6]:
schema = dict(
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

data = TimeSeries_datatype(data_df, **schema)

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


In [7]:
# configured, but holds no data yet
datamod = EncoderDecoderTimeSeriesDataModule(
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    train_val_test_split=(0.7, 0.15, 0.15),
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


In [8]:
forecaster = TFTForecaster(
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
    output_size=1,
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    datamodule=datamod,
    trainer=Trainer(max_epochs=2, accelerator="auto", log_every_n_steps=10),
)

forecaster.fit(data)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


TFTForecaster(datamodule=<pytorch_forecasting.data.data_module._encoder_decoder_data_module.EncoderDecoderTimeSeriesDataModule object at 0x71db08c16de0>,
              logging_metrics=[MAE(), SMAPE()], loss=MAE(),
              lr_scheduler='reduce_lr_on_plateau',
              lr_scheduler_params={'factor': 0.1, 'mode': 'min',
                                   'patience': 10},
              optimizer_params={'lr': 0.001},
              trainer=<lightning.pytorch.trainer.trainer.Trainer object at 0x71db08c16300>)

In [9]:
new_data = TimeSeries_datatype(data_df[data_df["series_id"] < 5], **schema)

preds = forecaster.predict(new_data)
preds.to_pandas().head()

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
💡 Tip: For seamless cloud uploads and versioning, tr

Predicting: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


,_series,_time_idx,y
0,0,30,0.442790
1,0,31,0.447545
2,0,32,0.455117
3,0,33,0.465580
4,0,34,0.479010


### 2.1 Which trainer is used

- A trainer given to `fit` wins over the one given to `__init__`.
- `fit` accepts a `Trainer` only, not trainer keywords, so there is one way to say it.
- `forecaster.trainer_` is the trainer that was actually used.

In [10]:
init_trainer = Trainer(max_epochs=1, logger=False, enable_checkpointing=False)
fit_trainer = Trainer(max_epochs=3, logger=False, enable_checkpointing=False)

forecaster = TFTForecaster(
    hidden_size=16,
    num_layers=1,
    attention_head_size=2,
    datamodule=datamod,
    trainer=init_trainer,
)

# no trainer in `fit` - the one from `__init__` is used
forecaster.fit(data)
print("used init trainer:", forecaster.trainer_ is init_trainer)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottlen

Training: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`


Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


used init trainer: True


In [11]:
# same forecaster, new trainer in `fit` - it overrides the one from `__init__`
forecaster.fit(data, trainer=fit_trainer)
print("used fit trainer:", forecaster.trainer_ is fit_trainer)
print("max_epochs:", forecaster.trainer_.max_epochs)

/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/skt

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottlen

Training: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`


Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=3` reached.


used fit trainer: True
max_epochs: 3


## 3. Minimal - without `Forecaster`

- The same three layers, now held by the user: datatype, datamodule, model.
- Data enters the datamodule as `timeseries_datatype=`.
- `metadata` is the only coupling between the layers, and is required by the model.
- It is available straight after `__init__`, no `setup` needed.

In [12]:
data = TimeSeries_datatype(
    data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
)
datamod = EncoderDecoderTimeSeriesDataModule(timeseries_datatype=data)

model = TFT(loss=MAE(), metadata=datamod.metadata)

Trainer(max_epochs=1).fit(model, datamod)

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The numbe

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


## 4. Maximal - without `Forecaster`

- `with_data` returns a new datamodule: same parameters, same fitted transforms, new data. It does not refit.
- Predictions come back as a dict of tensors, not as a `TimeSeries_datatype`.
- `from_tensors` and `to_pandas` bring them back to a data frame.
- Checkpointing and serialisation are the user's own on this path.

In [13]:
datamod = EncoderDecoderTimeSeriesDataModule(
    timeseries_datatype=TimeSeries_datatype(data_df, **schema),
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

model = TFT(
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
    metadata=datamod.metadata,
)

trainer = Trainer(max_epochs=2, accelerator="auto", devices=1, log_every_n_steps=10)
trainer.fit(model, datamod)

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/aryan/pytorch-forecasting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


In [14]:
new_datamod = datamod.with_data(
    TimeSeries_datatype(data_df[data_df["series_id"] < 5], **schema)
)
new_datamod.setup(stage="predict")

raw = model.predict(new_datamod.predict_dataloader())  # dict of tensors
raw["prediction"].shape

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/home/aryan/pytorch-forecasting/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:263: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
💡 Tip: For seamless cloud uploads and versioning, tr

Predicting: |          | 0/? [00:00<?, ?it/s]

torch.Size([95, 1])

In [15]:
# one row per forecast time step, one column per target
preds = TimeSeries_datatype.from_tensors({"y": raw["prediction"].reshape(-1, 1)})
preds.to_pandas().head()

/home/aryan/pytorch-forecasting/pytorch_forecasting/_proto/_timeseries_datatype.py:124: UserWarning: TimeSeries_datatype is a prototype of the reworked pytorch-forecasting data layer, for design testing only. It is not part of the public API and may change or disappear without warning. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


,_series,_time_idx,y0
0,0,0,0.573078
1,0,1,0.574299
2,0,2,0.575819
3,0,3,0.577434
4,0,4,0.579080
